# 01 Long-term series analysis

Hourly simulated flow of the three CFE runs (AORC, MRMS Pass 2, MRMS radar-only) against the USGS observed flow, over the whole scoring window and by season.

| Metric | Definition | Best value |
|---|---|---|
| NNSE | 1 / (2 - NSE) | 1 (0.5 = as good as the observed mean) |
| KGE | 1 - sqrt((r - 1)^2 + (sd_sim / sd_obs - 1)^2 + (mean_sim / mean_obs - 1)^2) | 1 (about -0.41 = as good as the observed mean) |
| PB | sum(sim - obs) / sum(obs) x 100 | 0 % (positive = too much flow) |
| NRMSE | RMSE / mean observed flow x 100 | 0 % |

Box plots: one box is the distribution over gages; box = quartiles, line = median, dashed whiskers = most extreme value within 1.5 box heights; points beyond are not drawn; a box needs at least 5 gages.

Maps: Albers Equal Area projection for the conterminous United States (EPSG:5070), aridity zones in the background.

**Paths.** Nothing to edit. The notebook finds the repository by itself and takes the data from the folder next to it (`cfe_evel/organized` and `cfe_evel/results`), so it runs the same from IDAS (`~/hpchome/Papers/cfe_evel/cfe_eval/analysis`) and from Argon. To use another data folder, set the environment variable `CFE_DATA_ROOT` before starting Jupyter.

**Output.** Figures (PNG, 300 dpi) and tables (CSV) are saved in `results/report/02_long_term_series/`, in one folder per type of analysis:

- `overview_tables/` medians and product comparison
- `full_period_by_climate/` box plots, three products side by side
- `by_season_and_climate/` box plots by season, one panel per product
- `maps/` score of each gage
- `maps_product_differences/` change of score between rain products
- `water_years/` medians year by year
- `by_basin_size/` box plots by drainage area

Next to each figure, `<name>_data.csv` holds the numbers behind it. `figure_list.csv` lists every figure and table with its caption, and `results/report/02_long_term_series.zip` packs everything for download. Set `FORMATS = ("png", "pdf")` in the first cell to also get vector files.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# find the repository (the folder that holds cfe_eval/config.py), whatever folder the notebook is opened from
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cfe_eval" / "config.py").exists())
sys.path.insert(0, str(REPO))
from cfe_eval import common as cm
from cfe_eval import config as cfg
from cfe_eval import maps, plots, report

%matplotlib inline
plots.slide_style()
pd.set_option("display.max_columns", 60, "display.width", 220)

# every figure and table goes to results/report/02_long_term_series/<type of analysis>/
SAVE = True                  # False = draw everything, write nothing
FORMATS = ("png",)           # ("png", "pdf") also writes vector files for a report
rep = report.Report("02_long_term_series", formats=FORMATS, save=SAVE)

# sentences reused in the captions
BOX = ("One box is the distribution over gages: box = quartiles, line = median, dashed whiskers = most extreme "
       "value within 1.5 box heights, values beyond are not drawn.")
MAP = "Albers Equal Area projection (EPSG:5070); background = aridity zones."

print("repository:", REPO)
print("data root :", cfg.DATA_ROOT)
print("tables    :", cfg.TABLES, "(found)" if cfg.TABLES.exists() else "(MISSING: run the evaluation jobs first)")
print("output    :", rep.root)

## Settings

`USE_QAQC = True` keeps only the gages that passed the quality control (notebook 00, file `results/tables/qaqc_sites.csv`). The notebook stops if that file does not exist. Set `USE_QAQC = False` to use all 818 gages.

In [ ]:
USE_QAQC = True
WHISKERS = 1.5      # box plot whiskers: 1.5 box heights; use (5, 95) for the 5th and 95th percentiles

keep = cm.qaqc_keep() if USE_QAQC else None
if USE_QAQC and keep is None:
    raise SystemExit("Quality control list not found. Run analysis/00_qaqc.ipynb first "
                     "(it writes results/tables/qaqc_sites.csv), or set USE_QAQC = False to use all gages.")

def load(name):
    """A results table, limited to the gages kept by the quality control."""
    df = cm.read_table(name)
    col = "site" if name == "gages" else "gage"
    return df if keep is None else df[df[col].isin(keep)].reset_index(drop=True)

gages = load("gages")
P = list(cfg.PRODUCTS)
print("quality control:", "NOT applied, all gages used" if keep is None else f"applied, {len(keep)} gages kept, {818 - len(keep)} removed")
gages.groupby("aridity_zone").size().reindex(cfg.ZONES).rename("gages").to_frame().T

In [ ]:
cont = load("continuous_metrics").merge(gages[["site", "lat", "lon"]].rename(columns={"site": "gage"}), on="gage")
full = cont[cont["period_type"] == "full"]
seas = cont[cont["period_type"] == "season"].rename(columns={"period": "season"})
METRICS = {"NNSE": "NNSE", "KGE": "KGE", "PBIAS_pct": "PB (%)", "NRMSE_pct": "NRMSE (%)"}
print(len(full), "rows for the full period,", len(seas), "rows by season")

## 1. Overview

Median over gages, full period, by rain product and climate type.

In [ ]:
overview = (full.groupby(["aridity_zone", "product"])[list(METRICS)].median()
            .reindex([(z, p) for z in cfg.ZONES for p in P]).round(3))
all_zones = full.groupby("product")[list(METRICS)].median().reindex(P).round(3)
all_zones.index = pd.MultiIndex.from_product([["All"], all_zones.index])
overview = pd.concat([overview, all_zones])
rep.table(overview, "overview_tables", "medians_full_period",
          f"Median over gages of the full-period scores by climate type and rain product ({len(gages)} gages).")

**Which rain product is better, gage by gage?** Share of gages (%) where the first product has the better score than the second. For PB the better score is the one closer to zero. 50 % means no difference.

In [ ]:
def share_better(metric, a, b, group=None):
    w = full.pivot(index="gage", columns="product", values=metric)
    if metric == "PBIAS_pct":
        better = w[a].abs() < w[b].abs()
    elif metric == "NRMSE_pct":
        better = w[a] < w[b]
    else:
        better = w[a] > w[b]
    better = better[w[[a, b]].notna().all(axis=1)]
    if group is None:
        return 100 * better.mean()
    return 100 * better.groupby(gages.set_index("site")[group].reindex(better.index)).mean()

pairs = [("MRMS_P2", "AORC"), ("MRMS_RO", "MRMS_P2"), ("MRMS_RO", "AORC")]
wins = pd.concat({f"{cfg.PRODUCT_LABEL[a]} better than {cfg.PRODUCT_LABEL[b]}":
                  pd.DataFrame({lab: share_better(m, a, b, "aridity_zone").reindex(cfg.ZONES).tolist()
                                + [share_better(m, a, b)] for m, lab in METRICS.items()},
                               index=cfg.ZONES + ["All"])
                  for a, b in pairs}).round(0)
rep.table(wins, "overview_tables", "share_of_gages_better",
          "Share of gages (%) where the first rain product scores better than the second, by climate type. "
          "For PB, better = closer to zero.")

## 2. Full period by climate type

One slide per metric: the three rain products side by side in each climate type.

In [ ]:
fig = plots.figure_zone_product(full, "NNSE", "NNSE", ref=0.5, ylim=(0, 1), whis=WHISKERS)
rep.figure(fig, "full_period_by_climate", "full_NNSE_by_climate",
           "Full-period NNSE by climate type for the three rain products. Line = 0.5 (as good as the observed mean). " + BOX)
fig = plots.figure_zone_product(full, "KGE", "KGE", ref=-0.41, whis=WHISKERS)
fig.axes[0].set_ylim(max(fig.axes[0].get_ylim()[0], -6), 1)
rep.figure(fig, "full_period_by_climate", "full_KGE_by_climate",
           "Full-period KGE by climate type for the three rain products. Line = -0.41 (as good as the observed mean); "
           "axis cut at -6. " + BOX)

The line in the KGE figure is -0.41, the KGE of a simulation equal to the observed mean (Knoben et al., 2019, https://doi.org/10.5194/hess-23-4323-2019). The axis is cut at -6.

In [ ]:
def symlog_pb(ax):
    """PB axis: linear between -100 and +100 %, logarithmic above."""
    ax.set_yscale("symlog", linthresh=100, linscale=1.5)
    ax.set_yticks([-100, -50, 0, 50, 100, 300, 1000, 3000, 10000])
    ax.set_yticklabels(["-100", "-50", "0", "50", "100", "300", "1000", "3000", "10000"])
    ax.minorticks_off()
    ax.set_ylim(-100, None)

def log_nrmse(ax):
    ax.set_yscale("log")
    ax.set_yticks([30, 100, 300, 1000, 3000, 10000, 30000])
    ax.set_yticklabels(["30", "100", "300", "1000", "3000", "10000", "30000"])
    ax.minorticks_off()

fig = plots.figure_zone_product(full, "PBIAS_pct", "PB (%)", ref=0, whis=WHISKERS)
symlog_pb(fig.axes[0])
rep.figure(fig, "full_period_by_climate", "full_PB_by_climate",
           "Full-period percent bias by climate type for the three rain products. Axis linear between -100 and 100 %, "
           "logarithmic above. " + BOX)
fig = plots.figure_zone_product(full, "NRMSE_pct", "NRMSE (%)", whis=WHISKERS)
log_nrmse(fig.axes[0])
rep.figure(fig, "full_period_by_climate", "full_NRMSE_by_climate",
           "Full-period NRMSE by climate type for the three rain products, logarithmic axis. " + BOX)

The PB axis is linear between -100 and +100 % and logarithmic above; the NRMSE axis is logarithmic. Both are needed because the dry climate types reach values far beyond the humid ones.

## 3. By season and climate type

One panel per rain product, one box per season. Seasonal values use all hours of that season in the scoring window, every year pooled.

In [ ]:
fig = plots.figure_by_product(seas, "NNSE", "NNSE", ref=0.5, ylim=(0, 1), whis=WHISKERS)
rep.figure(fig, "by_season_and_climate", "season_NNSE_by_climate",
           "NNSE by season and climate type, one panel per rain product. Line = 0.5. " + BOX)
fig = plots.figure_by_product(seas, "KGE", "KGE", ref=-0.41, whis=WHISKERS)
for ax in fig.axes:
    ax.set_ylim(max(ax.get_ylim()[0], -6), 1)
rep.figure(fig, "by_season_and_climate", "season_KGE_by_climate",
           "KGE by season and climate type, one panel per rain product. Line = -0.41; axis cut at -6. " + BOX)

In [ ]:
fig = plots.figure_by_product(seas, "PBIAS_pct", "PB (%)", ref=0, whis=WHISKERS)
for ax in fig.axes:
    symlog_pb(ax)
rep.figure(fig, "by_season_and_climate", "season_PB_by_climate",
           "Percent bias by season and climate type, one panel per rain product. Axis linear between -100 and 100 %, "
           "logarithmic above. " + BOX)
fig = plots.figure_by_product(seas, "NRMSE_pct", "NRMSE (%)", whis=WHISKERS)
for ax in fig.axes:
    log_nrmse(ax)
rep.figure(fig, "by_season_and_climate", "season_NRMSE_by_climate",
           "NRMSE by season and climate type, one panel per rain product, logarithmic axis. " + BOX)

The same PB figure on a plain linear axis, one slide per product, in the layout of the reference figures:

In [ ]:
for p in P:
    fig = plots.figure_single(seas, "PBIAS_pct", "PB (%)", p, ref=0, whis=WHISKERS)
    rep.figure(fig, "by_season_and_climate", f"season_PB_linear_{p}",
               f"Percent bias by season and climate type, {cfg.PRODUCT_LABEL[p]}, linear axis. " + BOX)

In [ ]:
med = (seas.groupby(["product", "aridity_zone", "season"])[list(METRICS)].median().round(2)
       .unstack("season").reindex(columns=cfg.SEASONS, level=1)
       .reindex(index=[(p, z) for p in P for z in cfg.ZONES]))
rep.table(med, "overview_tables", "medians_by_climate_season",
          "Median over gages of the seasonal scores by rain product, climate type and season.")

## 4. Maps

One circle per gage, coloured by class of the full-period score. To map a season instead, replace `full` by `seas[seas["season"] == "Summer"]`.

In [ ]:
MAPS = {
    "NNSE": dict(title="NNSE", bounds=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7], colors=maps.colors_from("RdYlBu", 7)),
    "KGE": dict(title="KGE", bounds=[-1, -0.41, 0, 0.25, 0.5, 0.75], colors=maps.colors_from("RdYlBu", 7)),
    "PBIAS_pct": dict(title="PB (%)", bounds=[-50, -20, 20, 50, 100, 300],
                      colors=["#8c510a", "#d8b365", "#ffffff", "#c7eae5", "#5ab4ac", "#01665e", "#003c30"]),
    "NRMSE_pct": dict(title="NRMSE (%)", bounds=[100, 150, 200, 300, 500, 1000], colors=maps.colors_from("YlOrRd", 7)),
}
for m, kw in MAPS.items():
    fig = maps.figure_metric_maps(full, m, **kw)
    rep.figure(fig, "maps", f"map_{m.replace('_pct', '')}",
               f"Full-period {kw['title']} of each gage for the three rain products. " + MAP)

### Difference between rain products

Change of the score when the rain product changes, gage by gage. Blue = the first product is better, red = worse.

In [ ]:
def difference_maps(metric, title, bounds, pairs=(("MRMS_P2", "AORC"), ("MRMS_RO", "MRMS_P2"))):
    w = full.pivot(index="gage", columns="product", values=metric)
    g = gages.set_index("site").reindex(w.index)
    colors = maps.colors_from("RdBu", len(bounds) + 1)
    fig, axes = plt.subplots(1, len(pairs), figsize=(20, 7.6))
    for ax, (a, b) in zip(axes, pairs):
        maps.metric_map(ax, g, (w[a] - w[b]).to_numpy(), bounds, colors, size=40)
        ax.set_title(f"{cfg.PRODUCT_LABEL[a]} minus {cfg.PRODUCT_LABEL[b]}")
    fig.tight_layout(w_pad=0.2, rect=(0, 0.15, 1, 1))
    from matplotlib.lines import Line2D
    h = [Line2D([], [], marker="o", linestyle="", markersize=11, markerfacecolor=c, markeredgecolor="black",
                markeredgewidth=0.5, label=lab) for c, lab in zip(colors, maps.class_labels(bounds))]
    fig.legend(handles=h, title=title, loc="lower center", bbox_to_anchor=(0.5, 0.065), ncol=len(h), frameon=False,
               columnspacing=1.2, handletextpad=0.2, title_fontproperties={"weight": "bold"})
    fig.legend(handles=maps.zone_handles(), loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False)
    d = pd.DataFrame({f"{a}_minus_{b}": w[a] - w[b] for a, b in pairs})
    fig.cfe_data = g[["lat", "lon", "aridity_zone"]].join(d).rename_axis("gage").reset_index()
    return fig

fig = difference_maps("KGE", "Difference in KGE", [-0.2, -0.1, -0.03, 0.03, 0.1, 0.2])
rep.figure(fig, "maps_product_differences", "map_difference_KGE",
           "Change of the full-period KGE of each gage between rain products. Blue = the first product is better. " + MAP)
fig = difference_maps("NNSE", "Difference in NNSE", [-0.1, -0.05, -0.02, 0.02, 0.05, 0.1])
rep.figure(fig, "maps_product_differences", "map_difference_NNSE",
           "Change of the full-period NNSE of each gage between rain products. Blue = the first product is better. " + MAP)

## 5. Year by year

Median over gages for each water year (October to September). Water year 2026 has only three months in the window and is left out.

In [ ]:
wy = cont[(cont["period_type"] == "water_year") & (cont["period"] != "2026")]
fig, axes = plt.subplots(1, 4, figsize=(20, 5.8))
for ax, (m, lab) in zip(axes, METRICS.items()):
    t = wy.groupby(["period", "product"])[m].median().unstack("product")
    for p in P:
        ax.plot(t.index, t[p], marker="o", lw=2.4, ms=8, color=plots.PRODUCT_COLORS[p][0], label=cfg.PRODUCT_LABEL[p])
    ax.set_title(lab)
    ax.set_xlabel("Water year")
    ax.grid(color="0.9")
fig.tight_layout(rect=(0, 0, 1, 0.9))
h, lab = axes[0].get_legend_handles_labels()
fig.legend(h, lab, loc="upper center", ncol=3, frameon=True, edgecolor="black", fancybox=False)
rep.figure(fig, "water_years", "water_year_medians",
           "Median over gages of each score by water year (October to September) for the three rain products.",
           data=wy.groupby(["period", "product"])[list(METRICS)].median().rename_axis(index={"period": "water_year"}))

## 6. By basin size

The same full-period boxes grouped by drainage area instead of climate type.

In [ ]:
fig = plots.figure_zone_product(full, "KGE", "KGE", ref=-0.41, whis=WHISKERS, group="area_class",
                                groups=cfg.AREA_LABELS, xlabel="Drainage area (km2)")
fig.axes[0].set_ylim(max(fig.axes[0].get_ylim()[0], -6), 1)
rep.figure(fig, "by_basin_size", "full_KGE_by_area",
           "Full-period KGE by drainage area class for the three rain products. Line = -0.41; axis cut at -6. " + BOX)
fig = plots.figure_zone_product(full, "NNSE", "NNSE", ref=0.5, ylim=(0, 1), whis=WHISKERS, group="area_class",
                                groups=cfg.AREA_LABELS, xlabel="Drainage area (km2)")
rep.figure(fig, "by_basin_size", "full_NNSE_by_area",
           "Full-period NNSE by drainage area class for the three rain products. Line = 0.5. " + BOX)

## Saved output

In [ ]:
rep.finish()

## Ideas to go further

- Months instead of seasons: `cont[cont["period_type"] == "month"]`.
- One season of one year: `period_type == "season_year"` (for example `2024-Summer`).
- The three KGE parts (correlation, variability ratio, bias ratio) are not stored yet; correlation `r` is. Adding the other two to step 2 would show which part drives the low KGE in the dry zones.
- Scores against distance to the nearest radar (`dist_nexrad_km` in the gages table), for the radar-only product.